# ENVIRONMENT

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain cohere

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['COHERE_API_KEY'] = os.getenv("COHERE_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

# RE - RANKING

Re-ranking scores chunks and places the most relevant ones at the top before sending them to the LLM (Cohere Re-rank).

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

splits = text_splitter.split_documents(blog_docs)

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=OpenAIEmbeddings())

In [ ]:
from langchain_community.llms import cohere
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

compressor = CohereRerank(
    model="rerank-v3.5",
    cohere_api_key=os.getenv("COHERE_API_KEY")
)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)
question = "What is task decomposition for LLM agents?"
compressed_docs = compression_retriever.invoke(question)

This block of code retrieves 10 relevant chunks and then uses the re-ranking model to extract the three most relevant chunks for the query.

In [ ]:
compressed_docs

### How Re-Ranking Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                           RE-RANKING FLOW                               │
└─────────────────────────────────────────────────────────────────────────┘

           ┌───────────────────┐
           │    User Query     │
           └─────────┬─────────┘
                     │
                     ▼
          ┌───────────────────────┐
          │    Retrieve top 10    │
          │   chunks from Chroma  │
          └───────────┬───────────┘
                     │
                     ▼
          ┌───────────────────────┐
          │   Cohere scores each  │
          │  chunk against query  │
          └───────────┬───────────┘
                     │
                     ▼
          ┌───────────────────────┐
          │     Top 3-4 most      │
          │   relevant returned   │
          └───────────┬───────────┘
                     │
                     ▼
           ┌───────────────────┐
           │    Sent to LLM    │
           └─────────┬─────────┘
                     │
                     ▼
           ┌───────────────────┐
           │      Answer       │
           └───────────────────┘
```

**Key Insights:**
- **Initial Retrieval**: First retrieves a larger set of candidates (e.g., top 10) from the vector store using standard similarity search.
- **Re-Ranking**: A specialized model (like Cohere) scores and reorders the chunks based on deeper semantic relevance to the query.
- **Refined Context**: Only the very best chunks (top 3-4) are ultimately sent to the LLM, reducing noise and context window usage while improving answer quality.

# CRAG (CORRECTIVE RAG)

Standard RAG blindly trusts retrieval. Whatever chunks are retrieved are sent directly to the LLM, but there is a possibility that these chunks are irrelevant. Bad context leads to a bad response. Therefore, after retrieving the chunks, we check their relevance. If they are relevant, we proceed to generate the answer. If they are not, we automatically perform a web search, retrieve the web results, and use them to generate the answer.

### How CRAG (Corrective RAG) Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                             CRAG FLOW                                   │
└─────────────────────────────────────────────────────────────────────────┘

                      ┌───────────────────┐
                      │    User Query     │
                      └─────────┬─────────┘
                                │
                                ▼
                    ┌───────────────────────┐
                    │    Retrieve chunks    │
                    │   from vector store   │
                    └───────────┬───────────┘
                                │
                                ▼
                    ┌───────────────────────┐
                    │  Check relevance of   │
                    │   retrieved chunks    │
                    └───────────┬───────────┘
                                │
                 ┌──────────────┴──────────────┐
                 │                             │
                 ▼                             ▼
       ┌───────────────────┐         ┌───────────────────┐
       │     Relevant      │         │   Not Relevant    │
       └─────────┬─────────┘         └─────────┬─────────┘
                 │                             │
                 ▼                             ▼
       ┌───────────────────┐         ┌─────────────────────┐
       │   Use retrieved   │         │   Search the Web    │
       │     chunks as     │         │ for better results  │
       │      context      │         └──────────┬──────────┘
       └─────────┬─────────┘                    │
                 │                              ▼
                 │                   ┌─────────────────────┐
                 │                   │   Use web results   │
                 │                   │     as context      │
                 │                   └──────────┬──────────┘
                 │                              │
                 └──────────────┬───────────────┘
                                │
                                ▼
                     ┌───────────────────┐
                     │  Generate Answer  │
                     │     using LLM     │
                     └───────────────────┘
```

**Key Insights:**
- **Relevance Check**: Unlike standard RAG, CRAG doesn't blindly trust retrieved chunks — it evaluates their relevance to the query before using them.
- **Web Fallback**: If retrieved chunks are deemed irrelevant, CRAG automatically falls back to web search to find better context.
- **Better Answers**: By ensuring only relevant context reaches the LLM (whether from the vector store or the web), CRAG produces more accurate and grounded responses.

# SELF - RAG

This checks whether the generated answer is good enough. If it is, we return it; if not, we regenerate it. The LLM grades its own output and retries retrieval if needed.

### How Self-RAG Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                          SELF-RAG FLOW                                  │
└─────────────────────────────────────────────────────────────────────────┘

                      ┌───────────────────┐
                      │    User Query     │
                      └─────────┬─────────┘
                                │
                                ▼
                    ┌───────────────────────┐
                    │    Retrieve chunks    │
                    │   from vector store   │
                    └───────────┬───────────┘
                                │
                                ▼
                    ┌───────────────────────┐
                    │    Generate Answer    │
                    │       using LLM       │
                    └───────────┬───────────┘
                                │
                                ▼
                    ┌───────────────────────┐
                    │    Is this answer     │
                    │     good enough?      │
                    │   (LLM self-grades)   │
                    └───────────┬───────────┘
                                │
                 ┌──────────────┴──────────────┐
                 │                             │
                 ▼                             ▼
       ┌───────────────────┐         ┌───────────────────┐
       │        Yes        │         │        No         │
       └─────────┬─────────┘         └─────────┬─────────┘
                 │                             │
                 ▼                             ▼
       ┌───────────────────┐         ┌─────────────────────┐
       │    Return the     │         │   Retrieve again    │
       │      answer       │         │ with refined query  │
       └───────────────────┘         └──────────┬──────────┘
                                                │
                                                ▼
                                     ┌─────────────────────┐
                                     │     Re-generate     │
                                     │  answer using LLM   │
                                     └──────────┬──────────┘
                                                │
                                                ▼
                                     ┌─────────────────────┐
                                     │   Return improved   │
                                     │       answer        │
                                     └─────────────────────┘
```

**Key Insights:**
- **Self-Grading**: The LLM evaluates its own generated answer for quality, relevance, and groundedness — it doesn't just generate and return blindly.
- **Retry Loop**: If the answer fails the quality check, the system retrieves new chunks and re-generates, improving the response iteratively.
- **Quality Assurance**: By adding this self-reflection step, Self-RAG ensures higher quality outputs compared to standard RAG, where the first generated answer is always returned.

# LONG CONTEXT

This solves a very critical problem. Suppose you retrieve 20 chunks and send all of them to the LLM. Research shows that LLMs do not read all context equally; they pay more attention to the beginning and the end of the context, while the middle gets ignored. This is known as the "lost-in-the-middle" problem. To address this, we place the most relevant chunks at the beginning and the end, and the less relevant chunks in the middle.